# Triplet Transformer — ICU Mortality Prediction

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.metrics import PrecisionRecallDisplay, RocCurveDisplay
from torch.utils.data import DataLoader, Dataset

from mortality_prediction.normalize_data import TRIPLET_VARS
from mortality_prediction.utils import DATA_DIR

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_VARS  = len(TRIPLET_VARS)   # 40 clinical variables
PAD_IDX = N_VARS               # index 40 reserved for padding tokens
print(f"PyTorch {torch.__version__}  |  device: {DEVICE}")
print(f"Clinical variables: {N_VARS}  |  PAD index: {PAD_IDX}")


## 1 — Load triplet data

In [ ]:
df_a = pd.read_parquet(os.path.join(DATA_DIR, "set_a_triplet.parquet"))
df_b = pd.read_parquet(os.path.join(DATA_DIR, "set_b_triplet.parquet"))
df_c = pd.read_parquet(os.path.join(DATA_DIR, "set_c_triplet.parquet"))

print(f"Set A: {df_a.shape}  |  Set B: {df_b.shape}  |  Set C: {df_c.shape}")
for name, df in [("A", df_a), ("B", df_b), ("C", df_c)]:
    pos = df.groupby("PatientID")["in_hospital_death"].first().mean()
    print(f"  Set {name}: {df['PatientID'].nunique()} patients  |  positive rate: {pos:.3f}")
df_a.head(4)


## 2 — Sequence lengths & max padding length

In [ ]:
# Compute per-patient event counts across all three splits
all_lengths = pd.concat([
    df_a.groupby("PatientID").size(),
    df_b.groupby("PatientID").size(),
    df_c.groupby("PatientID").size(),
])

print(f"Event counts across all patients:")
print(f"  min={all_lengths.min()}  p50={all_lengths.median():.0f}"
      f"  p95={all_lengths.quantile(.95):.0f}  max={all_lengths.max()}")

# Pad to the global maximum so every sequence fits without truncation
MAX_LEN = int(all_lengths.max())
print(f"\nMAX_LEN (padding target): {MAX_LEN}")


## 3 — Dataset & DataLoader

In [ ]:
class TripletDataset(Dataset):
    """
    Each sample is one patient's full event sequence, padded to MAX_LEN.

    Returns
    -------
    var_idx   : LongTensor  (MAX_LEN,)   — variable index; PAD_IDX for padding
    continuous: FloatTensor (MAX_LEN, 2) — (t, v) pairs; zeros for padding
    pad_mask  : BoolTensor  (MAX_LEN,)   — True at PAD positions (ignored by attention)
    label     : FloatTensor scalar
    """
    def __init__(self, df: pd.DataFrame, max_len: int, pad_idx: int):
        self.max_len = max_len
        self.pad_idx = pad_idx
        self.samples: list[tuple] = []

        for pid, group in df.sort_values("t").groupby("PatientID", sort=False):
            t   = group["t"].to_numpy(dtype=np.float32)
            z   = group["variable_idx"].to_numpy(dtype=np.int64)
            v   = group["v"].to_numpy(dtype=np.float32)
            lbl = float(group["in_hospital_death"].iloc[0])
            self.samples.append((t, z, v, lbl))

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        t, z, v, lbl = self.samples[idx]
        L = len(t)
        pad = self.max_len - L

        var_idx    = np.concatenate([z, np.full(pad, self.pad_idx, dtype=np.int64)])
        t_padded   = np.concatenate([t, np.zeros(pad, dtype=np.float32)])
        v_padded   = np.concatenate([v, np.zeros(pad, dtype=np.float32)])
        continuous = np.stack([t_padded, v_padded], axis=-1)   # (max_len, 2)
        pad_mask   = np.array([False] * L + [True] * pad, dtype=bool)

        return (
            torch.from_numpy(var_idx),
            torch.from_numpy(continuous),
            torch.from_numpy(pad_mask),
            torch.tensor(lbl, dtype=torch.float32),
        )


BATCH_SIZE = 32   # smaller batch: sequences are long

train_ds = TripletDataset(df_a, MAX_LEN, PAD_IDX)
val_ds   = TripletDataset(df_b, MAX_LEN, PAD_IDX)
test_ds  = TripletDataset(df_c, MAX_LEN, PAD_IDX)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)}  |  Val: {len(val_loader)}  |  Test: {len(test_loader)}")
print(f"Sample shapes — var_idx: {train_ds[0][0].shape}, continuous: {train_ds[0][1].shape}, mask: {train_ds[0][2].shape}")


## 4 — TripletTransformer model

In [ ]:
class TripletTransformer(nn.Module):
    """
    Transformer encoder over a variable-length sequence of clinical measurement
    triplets (t, z, v).

    Input encoding
    --------------
    Each event token is the SUM of:
      - a categorical embedding of the variable index z  (nn.Embedding)
      - a linear projection of the continuous pair (t, v) into d_model space

    Aggregation
    -----------
    Masked global average pooling: averages only over real (non-PAD) positions.

    Parameters
    ----------
    n_vars        : number of clinical variables (PAD token = n_vars)
    d_model       : embedding / hidden dimension (default 64)
    nhead         : attention heads (default 4; d_model must be divisible by nhead)
    num_layers    : number of TransformerEncoder layers (1 or 2)
    d_ff          : feed-forward inner dimension (default 128)
    dropout       : dropout rate (default 0.4)
    """
    def __init__(
        self,
        n_vars: int,
        d_model: int = 64,
        nhead: int = 4,
        num_layers: int = 1,
        d_ff: int = 128,
        dropout: float = 0.4,
    ):
        super().__init__()
        # Categorical embedding — index n_vars is the PAD token (embeddings zeroed)
        self.var_embed     = nn.Embedding(n_vars + 1, d_model, padding_idx=n_vars)
        # Continuous projection: (t, v) → d_model
        self.cont_proj     = nn.Linear(2, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_ff,
            dropout=dropout,
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, 1)

    def forward(
        self,
        var_idx: torch.Tensor,    # (B, L)  int64
        continuous: torch.Tensor, # (B, L, 2)  float32
        pad_mask: torch.Tensor,   # (B, L)  bool — True = ignore
    ) -> torch.Tensor:
        # Token embedding: sum categorical + continuous projections
        x = self.var_embed(var_idx) + self.cont_proj(continuous)  # (B, L, d_model)

        x = self.transformer(x, src_key_padding_mask=pad_mask)    # (B, L, d_model)

        # Masked global average pool — exclude PAD positions
        real = ~pad_mask                                           # (B, L)
        x = (x * real.unsqueeze(-1)).sum(dim=1)                   # (B, d_model)
        x = x / real.sum(dim=1, keepdim=True).clamp(min=1)

        return self.classifier(self.dropout(x)).squeeze(-1)        # (B,)


# Sanity check
_dummy_z    = torch.zeros(2, MAX_LEN, dtype=torch.long)
_dummy_cont = torch.zeros(2, MAX_LEN, 2)
_dummy_mask = torch.zeros(2, MAX_LEN, dtype=torch.bool)
for _layers in [1, 2]:
    _m = TripletTransformer(n_vars=N_VARS, num_layers=_layers)
    assert _m(_dummy_z, _dummy_cont, _dummy_mask).shape == (2,)
    _n = sum(p.numel() for p in _m.parameters() if p.requires_grad)
    print(f"  num_layers={_layers}  params={_n:,}")


## 5 — Model configurations & training utilities

In [ ]:
CONFIGS = [
    {"name": "Triplet-Transformer  1-layer", "num_layers": 1},
    {"name": "Triplet-Transformer  2-layer", "num_layers": 2},
]

MAX_EPOCHS   = 150
PATIENCE     = 15
LR           = 1e-3
WEIGHT_DECAY = 1e-4

y_train = torch.tensor(
    df_a.groupby("PatientID")["in_hospital_death"].first().to_numpy(dtype=np.float32)
)
pos_weight = torch.tensor([(y_train == 0).sum() / (y_train == 1).sum()]).to(DEVICE)


def evaluate(model: nn.Module, loader: DataLoader) -> tuple[float, float, np.ndarray]:
    model.eval()
    logits_all, labels_all = [], []
    with torch.no_grad():
        for z, cont, mask, y in loader:
            logits_all.append(model(z.to(DEVICE), cont.to(DEVICE), mask.to(DEVICE)).cpu())
            labels_all.append(y)
    logits = torch.cat(logits_all).numpy()
    labels = torch.cat(labels_all).numpy()
    probs  = torch.sigmoid(torch.from_numpy(logits)).numpy()
    return roc_auc_score(labels, probs), average_precision_score(labels, probs), probs


def train_model(cfg: dict) -> dict:
    model = TripletTransformer(
        n_vars=N_VARS,
        d_model=64,
        nhead=4,
        num_layers=cfg["num_layers"],
        d_ff=128,
        dropout=0.4,
    ).to(DEVICE)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    best_val_auroc = 0.0
    best_epoch     = 0
    best_state     = None
    patience_count = 0
    history        = []

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        total_loss = 0.0
        for z, cont, mask, y in train_loader:
            optimizer.zero_grad()
            loss = criterion(
                model(z.to(DEVICE), cont.to(DEVICE), mask.to(DEVICE)),
                y.to(DEVICE),
            )
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(y)

        val_auroc, val_auprc, _ = evaluate(model, val_loader)
        avg_loss = total_loss / len(train_loader.dataset)
        history.append({"epoch": epoch, "loss": avg_loss,
                        "val_auroc": val_auroc, "val_auprc": val_auprc})

        if val_auroc > best_val_auroc:
            best_val_auroc = val_auroc
            best_epoch     = epoch
            best_state     = {k: v.clone() for k, v in model.state_dict().items()}
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= PATIENCE:
                break

    model.load_state_dict(best_state)
    return {"model": model, "history": history,
            "best_epoch": best_epoch, "best_val_auroc": best_val_auroc}

print("Configurations:")
for cfg in CONFIGS:
    _m = TripletTransformer(n_vars=N_VARS, num_layers=cfg["num_layers"])
    _n = sum(p.numel() for p in _m.parameters() if p.requires_grad)
    print(f"  {cfg['name']:<38}  params={_n:,}")


## 6 — Train all configurations

In [ ]:
results = {}

for cfg in CONFIGS:
    print(f"\n{'─' * 56}")
    print(f"  {cfg['name']}")
    print(f"{'─' * 56}")
    print(f"  {'Epoch':>6}  {'Train loss':>10}  {'Val AuROC':>9}  {'Val AuPRC':>9}")

    res = train_model(cfg)
    results[cfg["name"]] = res

    for row in res["history"]:
        if row["epoch"] % 5 == 0 or row["epoch"] == 1:
            print(f"  {row['epoch']:>6}  {row['loss']:>10.4f}  {row['val_auroc']:>9.4f}  {row['val_auprc']:>9.4f}")

    n_epochs = len(res["history"])
    print(f"  → stopped at epoch {n_epochs}  |  best epoch {res['best_epoch']}  val AuROC={res['best_val_auroc']:.4f}")


## 7 — Training curves

In [ ]:
n_cfgs = len(results)
fig, axes = plt.subplots(1, n_cfgs, figsize=(7 * n_cfgs, 4))
if n_cfgs == 1:
    axes = [axes]

for ax, (name, res) in zip(axes, results.items()):
    hist = pd.DataFrame(res["history"])
    ax.plot(hist["epoch"], hist["val_auroc"], label="Val AuROC")
    ax.plot(hist["epoch"], hist["val_auprc"], label="Val AuPRC")
    ax.axvline(res["best_epoch"], color="red", linestyle="--",
               label=f"Best epoch ({res['best_epoch']})")
    ax.set_title(name)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Score")
    ax.legend(fontsize=9)

plt.suptitle("Validation metrics per configuration", fontsize=13)
plt.tight_layout()
plt.show()


## 8 — Validation set B comparison

In [ ]:
print("=" * 58)
print(f"{'Configuration':<38}  {'Val AuROC':>9}  {'Val AuPRC':>9}")
print("-" * 58)
for name, res in results.items():
    _, val_auprc, _ = evaluate(res["model"], val_loader)
    print(f"{name:<38}  {res['best_val_auroc']:>9.4f}  {val_auprc:>9.4f}")
print("=" * 58)


## 9 — Test set C evaluation (AuROC & AuPRC)

In [ ]:
print("=" * 58)
print(f"{'Configuration':<38}  {'AuROC':>7}  {'AuPRC':>7}")
print("-" * 58)
test_entries = []
for name, res in results.items():
    auroc, auprc, probs = evaluate(res["model"], test_loader)
    test_entries.append({"name": name, "auroc": auroc, "auprc": auprc, "probs": probs})
    print(f"{name:<38}  {auroc:>7.4f}  {auprc:>7.4f}")
print("=" * 58)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
y_test = torch.cat([y for _, _, _, y in test_loader]).numpy()
for entry in test_entries:
    RocCurveDisplay.from_predictions(y_test, entry["probs"],
                                     name=entry["name"], ax=axes[0])
    PrecisionRecallDisplay.from_predictions(y_test, entry["probs"],
                                            name=entry["name"], ax=axes[1])
axes[0].set_title("ROC curve — Test set C")
axes[1].set_title("Precision-Recall — Test set C")
plt.tight_layout()
plt.show()
